#Initialisation

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, trim, add_months, substring, to_date, expr
from pyspark.sql.types import StringType

In [0]:
RENAME_MAP = {
    "CID" : "customer_id",
    "BDATE" : "birth_date",
    "GEN" : "gender"
}

#Reading bronze table in

In [0]:
df = spark.read.table("workspace.bronze.erp_cust_az12_raw")

#Transfomations

##1. Trimming whitespace for our text columns

In [0]:
for field in df.schema:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

##2. Preparing keys for joining in modelling

In [0]:
df = df.withColumn("customer_key", substring(col("CID"),4,9999))

##3. Renaming the columns to something readable

In [0]:
for old_name,new_name in RENAME_MAP.items():
    df  = df.withColumnRenamed(old_name, new_name)

#Writting to the silver layer

In [0]:
(
    df.
    write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.erp_customer_details")
)